<a href="https://colab.research.google.com/github/RIAZ-28/RIAZ-28/blob/main/indi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!pip install ultralytics
import cv2
import numpy as np
from ultralytics import YOLO
from collections import defaultdict, deque

In [8]:
# History buffers to confirm BLINKING (prevents false positives)
left_hist  = defaultdict(lambda: deque(maxlen=5))
right_hist = defaultdict(lambda: deque(maxlen=5))

def detect_blinkers(vehicle_crop, vehicle_id):
    if vehicle_crop is None or vehicle_crop.size == 0:
        return "LEFT OFF", "RIGHT OFF"

    h, w, _ = vehicle_crop.shape

    # Split into left and right halves
    left_crop = vehicle_crop[:, :w//2]
    right_crop = vehicle_crop[:, w//2:]

    # Convert to HSV color space
    hsv_left = cv2.cvtColor(left_crop, cv2.COLOR_BGR2HSV)
    hsv_right = cv2.cvtColor(right_crop, cv2.COLOR_BGR2HSV)

    # Orange/Yellow indicator color range
    lower_orange = np.array([5, 100, 100])
    upper_orange = np.array([25, 255, 255])

    # Masks for left and right sides
    left_mask = cv2.inRange(hsv_left, lower_orange, upper_orange)
    right_mask = cv2.inRange(hsv_right, lower_orange, upper_orange)

    # Pixel count threshold for ON/OFF
    left_on = np.sum(left_mask > 0) > 50
    right_on = np.sum(right_mask > 0) > 50

    # Update tracking history
    left_hist[vehicle_id].append(left_on)
    right_hist[vehicle_id].append(right_on)

    # Decide status (blink if more than 2 recent ONs)
    left_status = "LEFT BLINK" if sum(left_hist[vehicle_id]) >= 2 else "LEFT OFF"
    right_status = "RIGHT BLINK" if sum(right_hist[vehicle_id]) >= 2 else "RIGHT OFF"

    return left_status, right_status


In [11]:
# Load YOLOv8 model
model = YOLO("yolov8n.pt")

# Set input and output video names
input_video = "testvideo (1).mp4"   # <-- Replace with your file
output_video = "output2.mp4"

cap = cv2.VideoCapture(input_video)

# Define output video writer
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(
    output_video,
    fourcc,
    cap.get(cv2.CAP_PROP_FPS),
    (int(cap.get(3)), int(cap.get(4)))
)

# YOLO tracking loop
for result in model.track(source=input_video, stream=True, persist=True):
    frame = result.orig_img
    boxes = result.boxes

    if boxes is None:
        out.write(frame)
        continue

    for box in boxes:
        cls = int(box.cls[0])

        # 2,3,5,7 = car, motorcycle, bus, truck
        if cls not in [2, 3, 5, 7]:
            continue
        if box.id is None:
          continue

        track_id = int(box.id[0])
        x1, y1, x2, y2 = map(int, box.xyxy[0])

        # Crop vehicle region
        vehicle_crop = frame[y1:y2, x1:x2]

        # Indicator detection
        left_status, right_status = detect_blinkers(vehicle_crop, track_id)
        # Draw bounding box
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

# Safe text position
        text_y = max(y1 - 10, 20)

        cv2.putText(
        frame,
        f"ID:{track_id} | {left_status} | {right_status}",
        (x1, text_y),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (0, 255, 255),
        2
    )


    out.write(frame)

cap.release()
out.release()




video 1/1 (frame 1/882) /content/testvideo (1).mp4: 384x640 (no detections), 250.2ms
video 1/1 (frame 2/882) /content/testvideo (1).mp4: 384x640 1 car, 127.2ms
video 1/1 (frame 3/882) /content/testvideo (1).mp4: 384x640 1 bus, 1 truck, 129.0ms
video 1/1 (frame 4/882) /content/testvideo (1).mp4: 384x640 1 car, 1 truck, 117.8ms
video 1/1 (frame 5/882) /content/testvideo (1).mp4: 384x640 (no detections), 133.3ms
video 1/1 (frame 6/882) /content/testvideo (1).mp4: 384x640 1 car, 118.6ms
video 1/1 (frame 7/882) /content/testvideo (1).mp4: 384x640 (no detections), 149.5ms
video 1/1 (frame 8/882) /content/testvideo (1).mp4: 384x640 1 car, 117.9ms
video 1/1 (frame 9/882) /content/testvideo (1).mp4: 384x640 (no detections), 118.2ms
video 1/1 (frame 10/882) /content/testvideo (1).mp4: 384x640 1 bus, 117.7ms
video 1/1 (frame 11/882) /content/testvideo (1).mp4: 384x640 1 bus, 1 truck, 132.9ms
video 1/1 (frame 12/882) /content/testvideo (1).mp4: 384x640 1 truck, 116.5ms
video 1/1 (frame 13/882) /c

In [12]:
# Store center positions per vehicle
track_centers = defaultdict(lambda: deque(maxlen=10))
def get_direction(track_id, center_x, center_y):
    track_centers[track_id].append((center_x, center_y))

    if len(track_centers[track_id]) < 5:
        return "UNKNOWN"

    dx = track_centers[track_id][-1][0] - track_centers[track_id][0][0]
    dy = track_centers[track_id][-1][1] - track_centers[track_id][0][1]
    box_width = x2 - x1
    threshold = 0.1*box_width
    if dx > threshold:
        return "RIGHT"
    elif dx < -threshold:
        return "LEFT"
    else:
        return "straight"
# Center of vehicle
cx = int((x1 + x2) / 2)
cy = int((y1 + y2) / 2)

direction = get_direction(track_id, cx, cy)
violation = "OK"

if direction == "LEFT" and left_status != "LEFT BLINK":
    violation = "LEFT TURN NO INDICATOR"

elif direction == "RIGHT" and right_status != "RIGHT BLINK":
    violation = "RIGHT TURN NO INDICATOR"
cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)

text_y = max(y1 - 10, 20)

cv2.putText(frame, f"ID:{track_id}", (x1, text_y),
            cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0,255,255), 2)

cv2.putText(frame, f"DIR:{direction}", (x1, text_y + 18),
            cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0,255,255), 2)

cv2.putText(frame, f"L:{left_status} R:{right_status}", (x1, text_y + 36),
            cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0,255,255), 2)

cv2.putText(frame, f"VIOL:{violation}", (x1, text_y + 54),
            cv2.FONT_HERSHEY_SIMPLEX, 0.55,
            (0,0,255) if violation != "OK" else (0,255,0), 2)
print("DONE! Saved output as:", output_video)


DONE! Saved output as: output2.mp4
